# Day05 下午个人项目：电商用户多维分析

**姓名：** 请填写  
**专题方向：** A / B / C / D / E

本Notebook由每名学生独立完成，并随个人项目仓库提交到GitHub。

> 请只修改标有 `TODO` 的区域，不要删除任务说明、检查点、结论区和提交检查。

## 一、实验目标与提交要求

你需要独立完成：

1. 读取并验收第4天清洗后的数据；
2. 计算公共基础指标；
3. 选择一个专题完成单维分析；
4. 完成至少一个双维度交叉分析；
5. 输出三个标准CSV报表；
6. 撰写至少3条结论、1条限制和1项建议；
7. 将Notebook和输出文件提交到个人GitHub仓库。

### 必须遵守的分析边界

- 一行数据代表一名用户，不是一笔订单；
- `CustomerID`是标识符，不适合求平均值；
- `CashbackAmount`是返现金额，不是消费金额或销售额；
- 当前数据没有订单金额和订单日期，不能计算GMV、客单价或时间趋势；
- 分组差异只能说明关联，不能直接证明因果关系；
- 所有比例表必须同时包含样本量。

## 二、专题方向

| 专题 | 推荐字段 | 参考业务问题 |
|---|---|---|
| A 用户生命周期 | `TenureGroup` | 不同生命周期用户的流失和订单行为有何差异？ |
| B 投诉与服务体验 | `Complain`、`SatisfactionScore` | 投诉、满意度与流失存在怎样的关联？ |
| C 品类与订单行为 | `PreferedOrderCat` | 不同偏好品类用户的规模和订单行为有何差异？ |
| D 支付与优惠行为 | `PreferredPaymentMode` | 支付偏好与优惠行为是否存在分组差异？ |
| E 城市与设备行为 | `CityTier`、`PreferredLoginDevice` | 城市、设备与用户活跃或流失有何关联？ |

请选择一个专题作为单维分析主线。双维分析可以在此基础上增加另一个业务维度。

## 任务0：个人配置与运行环境

In [17]:
from pathlib import Path
import pandas as pd
import numpy as np

try:
    from IPython.display import display
except ImportError:
    def display(obj):
        print(obj)


# =========================
# TODO：填写个人信息与专题
# =========================
STUDENT_NAME = "姚佳凤"
TOPIC = "A"


pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")


def find_workspace_root(start=None):
    """从当前目录向上寻找项目根目录。"""
    start = Path.cwd() if start is None else Path(start)

    for candidate in [start, *start.parents]:
        data_path = (
            candidate
            / "output"
            / "day04_project"
            / "ecommerce_customer_cleaned.csv"
        )

        if data_path.exists():
            return candidate

    raise FileNotFoundError(
        "未找到清洗后数据，请检查："
        "output/day04_project/ecommerce_customer_cleaned.csv"
    )


ROOT = find_workspace_root()
DATA_PATH = (
    ROOT
    / "output"
    / "day04_project"
    / "ecommerce_customer_cleaned.csv"
)
OUTPUT_DIR = ROOT / "output" / "day05_analysis"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


print("姓名：", STUDENT_NAME)
print("专题：", TOPIC)
print("输入数据：", DATA_PATH)
print("输出目录：", OUTPUT_DIR)

姓名： 姚佳凤
专题： A
输入数据： D:\desk\holiday study\ecommerce-user-analysis-seed\ecommerce-user-analysis-seed\output\day04_project\ecommerce_customer_cleaned.csv
输出目录： D:\desk\holiday study\ecommerce-user-analysis-seed\ecommerce-user-analysis-seed\output\day05_analysis


In [18]:
# 检查点0：个人信息与专题配置

assert STUDENT_NAME != "请填写姓名", "请填写STUDENT_NAME"
assert STUDENT_NAME.strip(), "姓名不能为空"

TOPIC = TOPIC.strip().upper()
assert TOPIC in {"A", "B", "C", "D", "E"}, \
    "TOPIC只能填写A、B、C、D或E"

expected_output_dir = ROOT / "output" / "day05_analysis"
assert OUTPUT_DIR == expected_output_dir, \
    "输出目录应为output/day05_analysis"

print("检查点0通过")
print("姓名：", STUDENT_NAME)
print("专题：", TOPIC)

检查点0通过
姓名： 姚佳凤
专题： A


### 检查点0完成标志

- [ ] 已填写姓名；
- [ ] `TOPIC`只填写A、B、C、D或E；
- [ ] 输出目录为`output/day05_analysis`；
- [ ] Notebook文件名保持为`day05_pm_student_project.ipynb`。

## 任务1：读取并验收数据（必做）

In [19]:
# 读取第4天清洗后的数据
df = pd.read_csv(DATA_PATH)

print("数据形状：", df.shape)
display(df.head())
print("\n字段类型：")
display(df.dtypes.to_frame("数据类型"))

数据形状： (5630, 22)


,CustomerID,Churn,Tenure,PreferredLoginDevice,CityTier,WarehouseToHome,PreferredPaymentMode,Gender,HourSpendOnApp,NumberOfDeviceRegistered,PreferedOrderCat,SatisfactionScore,MaritalStatus,NumberOfAddress,Complain,OrderAmountHikeFromlastYear,CouponUsed,OrderCount,DaySinceLastOrder,CashbackAmount,TenureGroup,IsMobileLogin
0,50001,1,4.00,Mobile Phone,3,6.00,Debit Card,Female,3.00,3,Laptop & Accessory,2,Single,9,1,11.00,1.00,1.00,5.00,159.93,0-6个月,1
1,50002,1,9.00,Mobile Phone,1,8.00,UPI,Male,3.00,4,Mobile Phone,3,Single,7,1,15.00,0.00,1.00,0.00,120.90,7-12个月,1
2,50003,1,9.00,Mobile Phone,1,30.00,Debit Card,Male,2.00,4,Mobile Phone,3,Single,6,1,14.00,0.00,1.00,3.00,120.28,7-12个月,1
3,50004,1,0.00,Mobile Phone,3,15.00,Debit Card,Male,2.00,4,Laptop & Accessory,5,Single,8,0,23.00,0.00,1.00,3.00,134.07,新用户,1
4,50005,1,0.00,Mobile Phone,1,12.00,Credit Card,Male,3.00,3,Mobile Phone,5,Single,3,0,11.00,1.00,1.00,3.00,129.60,新用户,1



字段类型：


,数据类型
CustomerID,int64
Churn,int64
Tenure,float64
PreferredLoginDevice,str
CityTier,int64
WarehouseToHome,float64
PreferredPaymentMode,str
Gender,str
HourSpendOnApp,float64
NumberOfDeviceRegistered,int64


In [20]:
# TODO 1：定义需要验收的核心字段
core_cols = [
    # 示例："CustomerID",
     "CustomerID", "Churn", "Tenure", "TenureGroup", "OrderCount",
    "CouponUsed", "CashbackAmount", "DaySinceLastOrder", "Complain",
    "PreferedOrderCat", "PreferredPaymentMode"
]


# TODO 2：完成数据验收表
# 至少包含：行数、列数、CustomerID重复数、核心字段缺失数、Churn取值
validation =  pd.Series({
    "行数": len(df),
    "列数": df.shape[1],
    "CustomerID重复数": int(df["CustomerID"].duplicated().sum()),
    "核心字段缺失数": int(df[core_cols].isna().sum().sum()),
    "Churn取值": sorted(df["Churn"].unique().tolist()),
}, name="验收结果"
)

# TODO 3：展示验收结果
display(validation)

行数                 5630
列数                   22
CustomerID重复数         0
核心字段缺失数               0
Churn取值          [0, 1]
Name: 验收结果, dtype: object

In [21]:
# 检查点1：数据结构与核心质量

assert isinstance(df, pd.DataFrame), "df还不是DataFrame"
assert df.shape == (5630, 22), "数据形状应为(5630, 22)"
assert df["CustomerID"].is_unique, "CustomerID应保持唯一"
assert set(df["Churn"].unique()) == {0, 1}, \
    "Churn应只包含0和1"

required_core_cols = {
    "CustomerID",
    "Churn",
    "TenureGroup",
    "OrderCount",
    "CouponUsed",
    "CashbackAmount",
    "DaySinceLastOrder",
}

assert required_core_cols.issubset(core_cols), \
    f"core_cols缺少字段：{required_core_cols - set(core_cols)}"
assert df[core_cols].notna().all().all(), \
    "核心分析字段仍存在缺失值"
assert validation is not None, "请完成validation验收表"

print("检查点1通过")

检查点1通过


### 数据粒度说明

请用一句话说明一行数据代表什么：

> TODO：
每条记录代表单个用户的相关信息
请说明为什么`CustomerID`不能作为普通连续数值求平均：

> TODO：
CustomerID只是客户唯一编号，不存在数值大小意义

## 任务2：公共基础指标（必做）

请构建`overall_metrics`，至少包含以下10项指标：

1. 用户数；
2. 流失人数；
3. 总体流失率；
4. 平均订单数；
5. 订单数中位数；
6. 平均优惠券使用次数；
7. 平均返现；
8. 平均App使用时长；
9. 平均满意度；
10. 平均距上次下单天数。

输出建议使用“指标、数值”两列的DataFrame。

In [22]:
# TODO：计算公共基础指标

overall_metrics = overall_metrics = overall_metrics = pd.DataFrame({
    "指标": ["用户数", "流失人数", "流失率", "平均订单数", "订单数中位数", "平均优惠券数", "平均返现", "平均App时长", "平均满意度", "平均距上次下单天数"],
    "数值": [
        df["CustomerID"].nunique(),
        df["Churn"].sum(),
        df["Churn"].mean(),
        df["OrderCount"].mean(),
        df["OrderCount"].median(),
        df["CouponUsed"].mean(),
        df["CashbackAmount"].mean(),
        df["HourSpendOnApp"].mean(),
        df["SatisfactionScore"].mean(),
        df["DaySinceLastOrder"].mean(),
    ]
})


# TODO：展示结果
display(overall_metrics)

,指标,数值
0,用户数,"5,630.00"
1,流失人数,948.00
2,流失率,0.17
3,平均订单数,2.96
4,订单数中位数,2.00
5,平均优惠券数,1.72
6,平均返现,177.22
7,平均App时长,2.93
8,平均满意度,3.07
9,平均距上次下单天数,4.46


In [23]:
# 检查点2：公共基础指标

assert isinstance(overall_metrics, pd.DataFrame), \
    "overall_metrics应为DataFrame"
assert len(overall_metrics) >= 10, \
    "公共基础指标至少包含10项"

# TODO：将变量赋值为你计算的总体流失率
overall_churn_rate = df['Churn'].mean()

assert overall_churn_rate is not None, \
    "请填写overall_churn_rate"
assert abs(overall_churn_rate - 0.16838365896980462) < 1e-8, \
    "总体流失率计算不正确"

print("检查点2通过")

检查点2通过


### 公共指标初步观察

请写出一条总体数据现象。此处只描述数据，不解释原因。

> TODO：请填写，例如“当前样本共有……名用户，总体流失率为……”。
当前样本共有5630名用户，总体流失率为0.168

## 任务3：单维专题分析（必做）

根据所选专题确定一个主分组字段，并使用`groupby + agg`完成命名聚合。

最低要求：

- 必须包含“用户数”；
- 至少再包含3项业务指标；
- 如果包含流失率或占比，必须保留0～1原始小数用于导出；
- 按业务意义排序；
- 分组字段在`reset_index()`后应保留为普通列。

In [24]:
topic_fields = {
    "A": {"TenureGroup"},
    "B": {"Complain", "SatisfactionScore"},
    "C": {"PreferedOrderCat"},
    "D": {"PreferredPaymentMode"},
    "E": {"CityTier", "PreferredLoginDevice"},
}

print("当前专题：", TOPIC)
print("可选主分组字段：", topic_fields[TOPIC])


# TODO 1：选择主分组字段
segment_field = "TenureGroup"


# TODO 2：使用groupby + agg完成命名聚合
segment_analysis = (df.groupby([segment_field], observed=True)
      .agg(
          用户数=("CustomerID", "nunique"),
          流失人数=("Churn", "sum"),
          流失率=("Churn", "mean"),
          平均订单数=("OrderCount", "mean"),
          平均返现=("CashbackAmount", "mean"),
          平均距上次下单天数=("DaySinceLastOrder", "mean"),
      )
      .reset_index()
)

# TODO 3：重置索引、按业务意义排序并展示
display(segment_analysis)

当前专题： A
可选主分组字段： {'TenureGroup'}


,TenureGroup,用户数,流失人数,流失率,平均订单数,平均返现,平均距上次下单天数
0,0-6个月,1459,417,0.29,2.68,164.48,4.00
1,13-24个月,1574,102,0.06,3.64,200.72,5.24
2,24个月以上,504,0,0.00,3.68,225.30,5.37
3,7-12个月,1585,157,0.10,2.66,161.48,4.33
4,新用户,508,272,0.54,1.89,142.44,2.88


In [25]:
# 检查点3：单维专题分析

assert segment_field in df.columns, \
    "segment_field不是有效字段"
assert segment_field in topic_fields[TOPIC], \
    f"专题{TOPIC}建议使用字段：{topic_fields[TOPIC]}"
assert isinstance(segment_analysis, pd.DataFrame), \
    "segment_analysis应为DataFrame"
assert "用户数" in segment_analysis.columns, \
    "专题分析表必须包含用户数"
assert len(segment_analysis) >= 2, \
    "专题分析至少应包含两个分组"
assert segment_analysis["用户数"].sum() == len(df), \
    "各分组用户数之和应等于总用户数"

print("检查点3通过")

检查点3通过


### 单维专题分析记录

**本专题要回答的业务问题：**

> TODO：不同生命周期的电商用户流失情况存在怎样差异

**数据现象：**

> TODO：必须写明群体、用户数、指标和具体数值。
> 随着用户生命周期变长，流失率整体呈现持续下降趋势。新用户流失率高达 54%，为所有分组最高；0-6 个月用户流失率 26%，7-12 个月流失率 10%，13-24 个月流失率 6%；24 个月以上老用户流失率为 0。同时可以观察到：用户 tenure 越长，平均订单数量、平均返现金额、平均满意度整体更高；新用户平均订单数、返现、满意度在各组最低。

**可能解释：**

> TODO：使用“相关、可能、值得关注、需验证”等有边界语言。

用户留存时长与用户活跃度、满意度可能存在正向相关，高频下单、满意度更高的用户更容易长期留存，流失风险更低。
新用户尚未建立使用习惯，对平台认同感弱，遇到较差体验更容易流失；随着持续使用，用户逐渐形成消费依赖，流失意愿下降，但该推论需验证。


## 任务4：双维度交叉分析（必做）

从以下维度中选择两个不同字段：

- `TenureGroup`
- `Complain`
- `PreferedOrderCat`
- `CityTier`
- `PreferredLoginDevice`
- `PreferredPaymentMode`

最低要求：

- 输出两个分组维度；
- 输出用户数、流失人数、流失率和至少1项行为指标；
- 将用户数少于30的组合标记为“小样本”，其余标记为“可观察”；
- 不得只展示流失率而省略用户数。

In [26]:
allowed_cross_fields = {
    "TenureGroup",
    "Complain",
    "PreferedOrderCat",
    "CityTier",
    "PreferredLoginDevice",
    "PreferredPaymentMode",
}


# TODO 1：选择两个不同维度
dim_1 = "TenureGroup"
dim_2 = "Complain"


# TODO 2：使用groupby + agg完成双维分析
cross_analysis = (
    df.groupby([dim_1, dim_2], observed=True)
      .agg(
          用户数=("CustomerID", "nunique"),
          流失人数=("Churn", "sum"),
          流失率=("Churn", "mean"),
          平均订单数=("OrderCount", "mean"),
      )
      .reset_index()
)
cross_analysis["投诉状态"] = cross_analysis["Complain"].map({0: "无投诉", 1: "有投诉"})
# TODO 3：新增“样本提示”列
# 用户数<30标记为“小样本”，否则标记为“可观察”
cross_analysis["样本提示"] = np.where(cross_analysis["用户数"] < 30, "小样本", "可观察")


# TODO 4：按流失率或用户数排序并展示
display(cross_analysis)

,TenureGroup,Complain,用户数,流失人数,流失率,平均订单数,投诉状态,样本提示
0,0-6个月,0,1027,187,0.18,2.63,无投诉,可观察
1,0-6个月,1,432,230,0.53,2.80,有投诉,可观察
2,13-24个月,0,1135,46,0.04,3.79,无投诉,可观察
3,13-24个月,1,439,56,0.13,3.27,有投诉,可观察
4,24个月以上,0,358,0,0.00,3.81,无投诉,可观察
5,24个月以上,1,146,0,0.00,3.38,有投诉,可观察
6,7-12个月,0,1192,74,0.06,2.66,无投诉,可观察
7,7-12个月,1,393,83,0.21,2.66,有投诉,可观察
8,新用户,0,314,133,0.42,1.75,无投诉,可观察
9,新用户,1,194,139,0.72,2.12,有投诉,可观察


In [27]:
# 检查点4：双维度交叉分析

assert dim_1 in allowed_cross_fields and dim_2 in allowed_cross_fields, \
    "两个分析维度必须来自允许字段"
assert dim_1 != dim_2, "两个分析维度不能相同"
assert isinstance(cross_analysis, pd.DataFrame), \
    "cross_analysis应为DataFrame"

required_cross_cols = {
    dim_1,
    dim_2,
    "用户数",
    "流失率",
    "样本提示",
}

assert required_cross_cols.issubset(cross_analysis.columns), \
    f"双维分析表缺少字段：{required_cross_cols - set(cross_analysis.columns)}"
assert cross_analysis["用户数"].sum() == len(df), \
    "双维组合用户数之和应等于总用户数"
assert set(cross_analysis["样本提示"]).issubset(
    {"小样本", "可观察"}
), "样本提示只能是“小样本”或“可观察”"

expected_sample_hint = np.where(
    cross_analysis["用户数"] < 30,
    "小样本",
    "可观察",
)
assert np.array_equal(
    cross_analysis["样本提示"].to_numpy(),
    expected_sample_hint,
), "样本提示与用户数阈值不一致"

print("检查点4通过")

检查点4通过


### 双维分析记录

**最值得关注的维度组合：**

> TODO：新用户：TenureGroup 有投诉：Complain

**该组合的用户数、流失率和比较对象：**

> TODO：用户数、流失率和投诉状态。

**是否存在小样本风险：**

> TODO：不存在，因为都是可观察状态。

**为什么不能直接写成因果结论：**

> TODO：只能观察到“投诉用户流失更高，但无法证明投诉直接造成流失，还有可能是体验感比原先差而引起投诉

## 任务5：输出统计报表（必做）

In [28]:
# 输出三个标准CSV文件

outputs = {
    "overall_metrics.csv": overall_metrics,
    "segment_analysis.csv": segment_analysis,
    "cross_analysis.csv": cross_analysis,
}

for filename, table in outputs.items():
    path = OUTPUT_DIR / filename
    table.to_csv(path, index=False, encoding="utf-8-sig")
    print("已输出：", path.relative_to(ROOT))

已输出： output\day05_analysis\overall_metrics.csv
已输出： output\day05_analysis\segment_analysis.csv
已输出： output\day05_analysis\cross_analysis.csv


In [29]:
# 检查点5：输出文件与回读验证

for filename, table in outputs.items():
    path = OUTPUT_DIR / filename

    assert path.exists(), f"缺少输出文件：{filename}"

    reloaded = pd.read_csv(path)

    assert reloaded.shape == table.shape, \
        f"{filename}回读后的形状与原表不一致"
    assert not any(
        str(col).startswith("Unnamed")
        for col in reloaded.columns
    ), f"{filename}包含多余索引列，请使用index=False导出"

    print(f"通过：{filename}，形状为{reloaded.shape}")

print("检查点5通过")

通过：overall_metrics.csv，形状为(10, 2)
通过：segment_analysis.csv，形状为(5, 7)
通过：cross_analysis.csv，形状为(10, 8)
检查点5通过


## 任务6：结论、限制与建议（必做）

### 结论1

在____用户中，____指标为____，与____相比____。对应证据表：____。

> TODO：在新用户中，有投诉群体流失率指标为72%，与同期无投诉新用户相比高出30%。对应证据表：cross_analysis。
。

### 结论2

> TODO：全生命周期分组内，存在投诉行为的用户流失率均高于同生命周期无投诉用户；其中随着用户在岗时长增加，投诉带来的流失增幅逐步减弱。新用户受投诉冲击最大，24 个月以上成熟用户无论是否投诉均无流失。

### 结论3

> TODO：用户生命周期越长，整体留存能力越强：新用户整体平均流失率最高，7-12 个月、13-24 个月无投诉用户流失率大幅回落，24 个月以上老用户实现零流失，长期用户忠诚度极强。
。

### 分析限制

至少写明一项当前数据不能支持的分析，或一项可能影响结论的限制。

> TODO：数据集为横截面静态数据，没有投诉发生时间、流失发生时间，无法确认投诉和流失的先后顺序，不能证明投诉是流失直接诱因。

### 运营建议与验证方式

提出一项与分析结果对应的建议，并说明还需要哪些数据或方法验证效果。

> TODO：建议：针对新用户投诉搭建优先处理通道，缩短投诉响应时长、配套小额补偿政策，重点挽回新客群体。
验证方案：需要新增数据：投诉提交时间、客服首次回复时长、投诉解决状态、后续 30 天用户活跃与下单数据。


## 拓展任务（选做）

In [30]:
# 可选方向：
# 1. 使用qcut或业务规则构建订单活跃度分层；
# 2. 将双维分析整理为第6天绘图使用的长表；
# 3. 对一个反直觉结果提出两种数据核查方法；
# 4. 增加一项不与必做任务重复的业务分析。

# TODO（选做）

## 最终检查：GitHub提交前验收

In [31]:
print(ROOT)

D:\desk\holiday study\ecommerce-user-analysis-seed\ecommerce-user-analysis-seed


In [32]:
required_files = [
    ROOT  / "notebooks" / "day05_pm_student_project.ipynb",
    OUTPUT_DIR / "overall_metrics.csv",
    OUTPUT_DIR / "segment_analysis.csv",
    OUTPUT_DIR / "cross_analysis.csv",
]

missing_files = [
    str(path.relative_to(ROOT))
    for path in required_files
    if not path.exists()
]

assert not missing_files, \
    f"提交内容不完整，缺少文件：{missing_files}"

for csv_path in required_files[1:]:
    check_df = pd.read_csv(csv_path)
    assert not any(
        str(col).startswith("Unnamed")
        for col in check_df.columns
    ), f"{csv_path.name}仍包含多余索引列"

print("本地提交文件检查通过")
print("请重启内核并从头运行Notebook，然后提交并推送到个人GitHub仓库。")

本地提交文件检查通过
请重启内核并从头运行Notebook，然后提交并推送到个人GitHub仓库。


### GitHub提交清单

- [ ] 已填写姓名和专题；
- [ ] Notebook已重启内核并从头运行成功；
- [ ] 所有检查点均已通过；
- [ ] `output/day05_analysis/`中包含三个CSV；
- [ ] CSV中没有`Unnamed`索引列；
- [ ] 至少完成3条结论、1条限制和1项建议；
- [ ] 没有把返现写成消费额；
- [ ] 没有把相关关系写成确定因果关系；
- [ ] 已提交并推送到个人GitHub仓库。

### 最终反思

1. 本次分析中最重要的数据发现是什么？
2. 哪个检查点最能帮助你发现错误？
3. 哪条结论最容易被误解为因果关系？
4. 如果增加一个字段，你最希望增加什么？
5. 第6天准备把哪张统计表转化为图表？为什么？